In [1]:
# ============================================================
# LAHTER 1: SEADISTUS + MUDELI LAADIMINE
# ============================================================
!git clone https://github.com/wazenmai/MIDI-BERT.git
%cd MIDI-BERT
!pip install miditoolkit==0.1.14 mido==1.2.10 numpy transformers seaborn matplotlib scikit-learn pandas tqdm music21

import sys, os, pickle, tempfile, glob
import torch
import torch.nn as nn
import numpy as np
import music21 as m21
import miditoolkit
from transformers import BertConfig
from google.colab import drive

if os.path.basename(os.getcwd()) != "MIDI-BERT":
    %cd MIDI-BERT
sys.path.append(os.getcwd())

!mkdir -p Data/CP_data/tmp
!mkdir -p data_creation/prepare_data/dict
!python data_creation/prepare_data/dict/make_dict.py

from data_creation.prepare_data.model import CP
from data_creation.prepare_data.utils import DEFAULT_DURATION_BINS
from MidiBERT.model import MidiBert
np.int = int

drive.mount('/content/drive')

# --- Sõnastik ---
dict_path = 'data_creation/prepare_data/dict/CP.pkl'
with open(dict_path, 'rb') as f:
    e2w, w2e = pickle.load(f)

# --- MidiBERT konfiguratsioon ---
config = BertConfig(
    max_position_embeddings=512,
    position_embedding_type='relative_key_query',
    hidden_size=768, num_attention_heads=12, num_hidden_layers=12,
    pad_token_id=0, output_attentions=True, attn_implementation="eager"
)
config._attn_implementation = "eager"
midibert_base = MidiBert(bertConfig=config, e2w=e2w, w2e=w2e)

# --- FolkClassifier (TÄPSELT SAMA arhitektuur kui treenimisel) ---
class FolkClassifier(nn.Module):
    def __init__(self, pretrained_midibert, num_classes, mask_bar_tokens=False):
        super().__init__()
        self.midibert = pretrained_midibert
        self.dropout = nn.Dropout(0.5)
        self.classifier = nn.Linear(768, num_classes)
        self.mask_bar_tokens = mask_bar_tokens

    def forward(self, input_ids, attn_mask, return_attentions=False):
        if self.mask_bar_tokens:
            input_ids = input_ids.clone()
            input_ids[..., 0] = 0

        embs = [self.midibert.word_emb[i](input_ids[..., i])
                for i, _ in enumerate(self.midibert.classes)]
        emb_linear = self.midibert.in_linear(torch.cat([*embs], dim=-1))

        outputs = self.midibert.bert(
            inputs_embeds=emb_linear, attention_mask=attn_mask,
            output_hidden_states=True, output_attentions=True
        )

        hidden = outputs.last_hidden_state
        mask = attn_mask.unsqueeze(-1).float()
        summed = (hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        pooled_output = summed / counts

        logits = self.classifier(self.dropout(pooled_output))
        if return_attentions:
            return logits, outputs.attentions
        return logits

# --- Laadi treenitud mudel ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
target_labels = ['Eesti', 'Soome']

model = FolkClassifier(midibert_base, len(target_labels)).to(device)
model_path = "/content/drive/MyDrive/LÕPUTÖÖ/final_folk_model.pt"
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# --- CP tokenizer ---
cp_model = CP(dict=dict_path)

PAD_BAR_IDX = e2w['Bar']['Bar <PAD>']
TARGET_TPB = 480

print(f"Mudel laetud: {model_path}")
print(f"Seade: {device}")
print(f"Klassid: {target_labels}")
print("VALMIS!")

Cloning into 'MIDI-BERT'...
remote: Enumerating objects: 1258, done.
remote: Counting objects: 100% (392/392), done.
remote: Compressing objects: 100% (254/254), done.
remote: Total 1258 (delta 172), reused 343 (delta 134), pack-reused 866 (from 1)
Receiving objects: 100% (1258/1258), 125.78 MiB | 6.06 MiB/s, done.
Resolving deltas: 100% (620/620), done.
Updating files: 100% (164/164), done.
/content/MIDI-BERT
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.9 MB/s eta 0:00:00
  Created wheel for miditoolkit: filename=miditoolkit-0.1.14-py3-none-any.whl size=19508 sha256=19c94815e499bd230350b0301ef31b9b1449d67e3a86128d316740fa7ae83d57
  Stored in directory: /root/.cache/pip/wheels/f0/6e/d3/005b67a60d4a610bc9de6a19f02d205e7ce19972725138fff4
Successfully built miditoolkit
{'Bar': {'Bar New': 0, 'Bar Continue': 1, 'Bar <PAD>': 2, 'Bar <MASK>': 3}, 'Position': {'Position 1/16': 0, 'Position 2/16': 1, 'Position 3/16': 2, 'Position 4/16': 3,

In [33]:
# ============================================================
# LAHTER 2: SAMM 3 — ANALÜÜSIFUNKTSIOONID
# ============================================================
import glob

def normalize_tpb(midi_path, target_tpb=480):
    """Kopeerib MIDI ajutisse faili ja normaliseerib ticks_per_beat."""
    midi = miditoolkit.MidiFile(midi_path)
    if midi.ticks_per_beat == target_tpb:
        return midi_path

    scale = target_tpb / midi.ticks_per_beat
    for inst in midi.instruments:
        for note in inst.notes:
            note.start = int(round(note.start * scale))
            note.end = int(round(note.end * scale))
        for cc in inst.control_changes:
            cc.time = int(round(cc.time * scale))
        for pb in inst.pitch_bends:
            pb.time = int(round(pb.time * scale))
    for tc in midi.tempo_changes:
        tc.time = int(round(tc.time * scale))
    for ts in midi.time_signature_changes:
        ts.time = int(round(ts.time * scale))
    for ks in midi.key_signature_changes:
        ks.time = int(round(ks.time * scale))

    midi.ticks_per_beat = target_tpb
    tmp = tempfile.NamedTemporaryFile(suffix='.mid', delete=False)
    midi.dump(tmp.name)
    return tmp.name


def safe_parse_xml(path):
    """Parsib XML faili, vajadusel eemaldab vigased kordusmärgid."""
    try:
        score = m21.converter.parse(path)
        # Testi kas write töötab (see triggerib repeat expansion)
        tmp_test = tempfile.NamedTemporaryFile(suffix='.mid', delete=False)
        score.write('midi', fp=tmp_test.name)
        os.remove(tmp_test.name)
        return score
    except:
        pass

    print("  [HOIATUS] Eemaldan vigased kordusmärgid...")
    score = m21.converter.parse(path, forceSource=True)
    for measure in score.recurse().getElementsByClass('Measure'):
        if isinstance(getattr(measure, 'leftBarline', None), m21.bar.Repeat):
            measure.leftBarline = m21.bar.Barline('regular')
        if isinstance(getattr(measure, 'rightBarline', None), m21.bar.Repeat):
            measure.rightBarline = m21.bar.Barline('regular')
    for el in list(score.recurse().getElementsByClass(
            ['RepeatBracket', 'RepeatExpression', 'DalSegno', 'DaCapo', 'Coda', 'Segno'])):
        try:
            el.activeSite.remove(el)
        except:
            pass
    return score


def compute_rollout(attentions, device):
    """Attention Rollout: kumulatiivne tähelepanu läbi kõigi kihtide."""
    seq_len = attentions[0].size(-1)
    result = torch.eye(seq_len).to(device)
    for attention in attentions:
        attn_weights = attention[0].mean(dim=0)
        attn_weights = attn_weights + torch.eye(seq_len).to(device)
        attn_weights = attn_weights / attn_weights.sum(dim=-1, keepdim=True)
        result = torch.matmul(attn_weights, result)
    return result


def analyze_folk_midi(original_path, model, cp_model, device,
                      target_labels, max_len=256):
    """
    Samm 3: originaalfail → normaliseeritud MIDI → tokenid → mudel → attention rollout.
    """
    is_xml = original_path.lower().endswith(('.xml', '.mxl', '.musicxml'))
    if is_xml:
        score = safe_parse_xml(original_path)
        tmp_midi = tempfile.NamedTemporaryFile(suffix='.mid', delete=False)
        score.write('midi', fp=tmp_midi.name)
        midi_path = tmp_midi.name
    else:
        midi_path = original_path

    norm_path = normalize_tpb(midi_path, TARGET_TPB)

    midi = miditoolkit.MidiFile(norm_path)
    midi.tempo_changes = [miditoolkit.TempoChange(tempo=120, time=0)]
    midi.dump(norm_path)

    try:
        input_ids, _ = cp_model.prepare_data([norm_path], task='melody', max_len=max_len)
    except Exception as e:
        print(f"[VIGA] Tokeniseerimine ebaõnnestus: {e}")
        return None

    if len(input_ids) == 0:
        print("[VIGA] Failis pole piisavalt noote.")
        return None

    input_tensor = torch.tensor(input_ids[0], dtype=torch.long).unsqueeze(0).to(device)
    attn_mask = (input_tensor[:, :, 0] != PAD_BAR_IDX).float().to(device)

    with torch.no_grad():
        logits, attentions = model(input_tensor, attn_mask, return_attentions=True)
        probs = torch.nn.functional.softmax(logits, dim=1)
        predicted_class = torch.argmax(probs, dim=1).item()

    rollout_matrix = compute_rollout(attentions, device)
    token_importance = rollout_matrix.mean(dim=0).cpu().numpy()

    ti_min, ti_max = token_importance.min(), token_importance.max()
    if ti_max > ti_min:
        norm_importance = (token_importance - ti_min) / (ti_max - ti_min)
    else:
        norm_importance = np.zeros_like(token_importance)

    segment = input_ids[0]

    midi_info = miditoolkit.MidiFile(norm_path)
    if midi_info.time_signature_changes:
        ts = midi_info.time_signature_changes[0]
        beats_per_bar = ts.numerator * (4.0 / ts.denominator)
    else:
        beats_per_bar = 4.0

    token_notes = []
    current_bar = -1

    for i, token in enumerate(segment):
        b_idx, p_idx, pt_idx, d_idx = int(token[0]), int(token[1]), int(token[2]), int(token[3])

        bar_str = w2e['Bar'].get(b_idx, '')
        pos_str = w2e['Position'].get(p_idx, '')
        pitch_str = w2e['Pitch'].get(pt_idx, '')

        if '<PAD>' in bar_str or '<MASK>' in bar_str:
            continue
        if '<PAD>' in pitch_str or '<MASK>' in pitch_str:
            continue

        if bar_str == 'Bar New':
            current_bar += 1

        try:
            pos_part = pos_str.split(' ')[1]
            pos_val = int(pos_part.split('/')[0]) - 1
            offset = pos_val * (beats_per_bar / 16.0)
        except:
            offset = 0.0

        try:
            pitch = int(pitch_str.split(' ')[1])
        except:
            continue

        abs_start = (current_bar * beats_per_bar) + offset

        token_notes.append({
            'pitch': pitch,
            'abs_start': abs_start,
            'importance': float(norm_importance[i]),
            'token_idx': i
        })

    if is_xml and os.path.exists(tmp_midi.name):
        os.remove(tmp_midi.name)
    if norm_path != midi_path and os.path.exists(norm_path):
        os.remove(norm_path)

    print(f"Ennustus: {target_labels[predicted_class]} "
          f"({probs[0][predicted_class].item()*100:.1f}%)")
    for i, label in enumerate(target_labels):
        bar = '█' * int(probs[0][i].item() * 30)
        print(f"  {label:>6s}: {probs[0][i].item()*100:5.1f}% {bar}")
    print(f"Tokeneid (mitte-PAD): {len(token_notes)}")

    return {
        'probs': probs,
        'predicted_class': predicted_class,
        'token_notes': token_notes,
        'beats_per_bar': beats_per_bar
    }


test_file = "/content/drive/MyDrive/LÕPUTÖÖ/rahvalaulud/test/no name.mid"
if os.path.exists(test_file):
    print(f"\n=== TEST: {os.path.basename(test_file)} ===")
    result = analyze_folk_midi(test_file, model, cp_model, device, target_labels)
else:
    print(f"Testfaili ei leitud: {test_file}")

print("\nSamm 3 VALMIS!")


=== TEST: no name.mid ===


100%|██████████| 1/1 [00:00<00:00, 109.76it/s]

Ennustus: Soome (99.0%)
   Eesti:   1.0% 
   Soome:  99.0% █████████████████████████████
Tokeneid (mitte-PAD): 124

Samm 3 VALMIS!


In [39]:
# ============================================================
# LAHTER 3: SAMM 4+5 — JOONDAMINE + VÄRVITUD PARTITUUR
# ============================================================
import matplotlib
import matplotlib.colors as mcolors
from scipy.spatial.distance import cdist

ATTENTION_THRESHOLD = 0.3
VELOCITY_MIN = 30
VELOCITY_MAX = 127
CMAP_NAME = 'OrRd'
REMOVE_HARMONY = True     # True = eemalda, False = jäta alles
FORCE_VIOLIN = True       # True = sunni viiul, False = originaalpill


def map_attention_to_original(original_path, token_notes, beats_per_bar):
    """
    Samm 4: loeb originaalfaili, joondab tokenid originaalnootidega.
    """
    try:
        score = m21.converter.parse(original_path)
    except:
        score = safe_parse_xml(original_path)

    original_notes = []
    for n in score.recurse().notes:
        if n.isChord:
            for p in n.pitches:
                original_notes.append({
                    'obj': n,
                    'pitch': p.midi,
                    'offset': float(n.offset + n.activeSite.offset
                                    if n.activeSite and hasattr(n.activeSite, 'offset')
                                    else n.offset),
                    'matched': False
                })
        else:
            original_notes.append({
                'obj': n,
                'pitch': n.pitch.midi,
                'offset': float(n.offset + n.activeSite.offset
                                if n.activeSite and hasattr(n.activeSite, 'offset')
                                else n.offset),
                'matched': False
            })

    if not original_notes or not token_notes:
        print(f"[HOIATUS] Originaalnoote: {len(original_notes)}, tokeneid: {len(token_notes)}")
        return score, original_notes, {}

    max_time_orig = max(n['offset'] for n in original_notes) or 1.0
    max_time_token = max(tn['abs_start'] for tn in token_notes) or 1.0

    orig_coords = np.array([
        [n['pitch'], n['offset'] / max_time_orig * 100]
        for n in original_notes
    ])
    token_coords = np.array([
        [tn['pitch'], tn['abs_start'] / max_time_token * 100]
        for tn in token_notes
    ])

    distances = cdist(token_coords, orig_coords)

    token_to_orig = {}
    used_orig = set()
    sorted_tokens = sorted(range(len(token_notes)),
                           key=lambda i: -token_notes[i]['importance'])

    for t_idx in sorted_tokens:
        candidates = []
        for o_idx in range(len(original_notes)):
            if o_idx in used_orig:
                continue
            pitch_diff = abs(token_notes[t_idx]['pitch'] - original_notes[o_idx]['pitch'])
            time_dist = distances[t_idx][o_idx]
            if pitch_diff <= 1:
                candidates.append((time_dist, o_idx))

        if candidates:
            candidates.sort()
            best_orig = candidates[0][1]
            token_to_orig[t_idx] = best_orig
            used_orig.add(best_orig)
            original_notes[best_orig]['matched'] = True

    print(f"Joondatud: {len(token_to_orig)}/{len(token_notes)} tokenit "
          f"→ {len(used_orig)}/{len(original_notes)} originaalnooti")

    orig_importance = {}
    for t_idx, o_idx in token_to_orig.items():
        orig_importance[o_idx] = token_notes[t_idx]['importance']

    return score, original_notes, orig_importance


def build_folk_score(score, original_notes, orig_importance,
                     threshold=ATTENTION_THRESHOLD, cmap_name=CMAP_NAME):
    """
    Samm 5: värvib noodid ja seab velocity.
    """
    cmap = matplotlib.colormaps[cmap_name]

    colored_count = 0
    black_count = 0
    grey_count = 0

    note_obj_importance = {}
    for o_idx, note_info in enumerate(original_notes):
        obj = note_info['obj']
        if o_idx in orig_importance:
            note_obj_importance[id(obj)] = orig_importance[o_idx]

    for n in score.recurse().notes:
        obj_id = id(n)

        if obj_id in note_obj_importance:
            imp = note_obj_importance[obj_id]

            if imp >= threshold:
                scaled = (imp - threshold) / (1.0 - threshold)
                rgba = cmap(0.3 + scaled * 0.7)
                color_hex = mcolors.to_hex(rgba)
                velocity = int(VELOCITY_MIN + scaled * (VELOCITY_MAX - VELOCITY_MIN))
                colored_count += 1
            else:
                color_hex = '#000000'
                velocity = VELOCITY_MIN
                black_count += 1
        else:
            color_hex = '#888888'
            velocity = VELOCITY_MIN
            grey_count += 1

        n.style.color = color_hex
        n.volume = m21.volume.Volume(velocity=velocity)

        if n.isChord:
            for sub_note in n.notes:
                sub_note.style.color = color_hex
                sub_note.volume = m21.volume.Volume(velocity=velocity)

    # --- INSTRUMENT ---
    if FORCE_VIOLIN:
        for part in score.parts:
            for inst in list(part.recurse().getElementsByClass('Instrument')):
                inst.midiProgram = 40
            if not list(part.recurse().getElementsByClass('Instrument')):
                part.insert(0, m21.instrument.Violin())

    # --- HARMONY/CHORD sümbolid ---
    if REMOVE_HARMONY:
        for h in list(score.recurse().getElementsByClass(['ChordSymbol', 'Harmony'])):
            try:
                h.activeSite.remove(h)
            except:
                pass
        print("Harmony sümbolid eemaldatud (REMOVE_HARMONY=True)")
    else:
        for h in score.recurse().getElementsByClass('ChordSymbol'):
            h.style.color = '#000000'
        print("Harmony sümbolid alles jäetud (REMOVE_HARMONY=False)")

    print(f"Värvitud: {colored_count} värvilist, {black_count} musta, {grey_count} halli")
    print(f"Velocity vahemik: {VELOCITY_MIN}–{VELOCITY_MAX}")
    return score


def export_attention_musicxml(original_path, result, output_dir):
    """
    Täistsükkel: originaalfail + mudeli tulemus → värvitud MusicXML.
    """
    base_name = os.path.splitext(os.path.basename(original_path))[0]
    output_path = os.path.join(output_dir, f"{base_name}_rollout.musicxml")
    os.makedirs(output_dir, exist_ok=True)

    print(f"\nTöötlen: {os.path.basename(original_path)}")

    score, orig_notes, orig_imp = map_attention_to_original(
        original_path, result['token_notes'], result['beats_per_bar']
    )

    score = build_folk_score(score, orig_notes, orig_imp)

    pred = target_labels[result['predicted_class']]
    prob = result['probs'][0][result['predicted_class']].item() * 100
    if score.metadata is None:
        score.insert(0, m21.metadata.Metadata())
    score.metadata.title = f"{base_name} — {pred} ({prob:.0f}%)"

    score.write('musicxml', output_path, makeNotation=False)
    print(f"Salvestatud: {output_path}")
    return score


# === TEST ===
test_file = "/content/drive/MyDrive/LÕPUTÖÖ/rahvalaulud/eesti/2.xml"
output_dir = "/content/drive/MyDrive/LÕPUTÖÖ/rollout_tulemused"

if os.path.exists(test_file) and 'result' in dir():
    test_score = export_attention_musicxml(test_file, result, output_dir)
    print("\nAva MuseScore'is ja kontrolli!")
else:
    print("Käivita enne lahter 2.")

print("\nSamm 4+5 VALMIS!")


Töötlen: 2.xml
Joondatud: 42/96 tokenit → 42/67 originaalnooti
Harmony sümbolid eemaldatud (REMOVE_HARMONY=True)
Värvitud: 42 värvilist, 0 musta, 9 halli
Velocity vahemik: 30–127
Salvestatud: /content/drive/MyDrive/LÕPUTÖÖ/rollout_tulemused/2_rollout.musicxml

Ava MuseScore'is ja kontrolli!

Samm 4+5 VALMIS!


In [40]:
# ============================================================
# LAHTER 4: TESTI 6 FAILI (3 Eesti XML + 3 Soome MIDI)
# ============================================================
import random
random.seed(42)

eesti_dir = "/content/drive/MyDrive/LÕPUTÖÖ/rahvalaulud/eesti"
soome_dir = "/content/drive/MyDrive/LÕPUTÖÖ/rahvalaulud/soome"
output_dir = "/content/drive/MyDrive/LÕPUTÖÖ/rollout_tulemused"
os.makedirs(output_dir, exist_ok=True)

# Leia failid
eesti_files = sorted(
    glob.glob(os.path.join(eesti_dir, "*.xml")) +
    glob.glob(os.path.join(eesti_dir, "*.mxl"))
)
soome_files = sorted(glob.glob(os.path.join(soome_dir, "*.mid")))

print(f"Eesti faile saadaval: {len(eesti_files)}")
print(f"Soome faile saadaval: {len(soome_files)}")

# Vali 3 igast klassist
eesti_sample = random.sample(eesti_files, min(3, len(eesti_files)))
soome_sample = random.sample(soome_files, min(3, len(soome_files)))

test_files = (
    [(f, "Eesti") for f in eesti_sample] +
    [(f, "Soome") for f in soome_sample]
)

# Analüüsi ja ekspordi kõik
all_results = {}

for path, true_label in test_files:
    name = os.path.basename(path)
    print(f"\n{'='*60}")
    print(f"{true_label}: {name}")
    print('='*60)

    result = analyze_folk_midi(path, model, cp_model, device, target_labels)

    if result is None:
        print(f"  [VAHELE JÄETUD]")
        continue

    # Kas ennustus õige?
    pred_label = target_labels[result['predicted_class']]
    correct = "✓" if pred_label == true_label else "✗"
    print(f"  Tegelik: {true_label} | Ennustus: {pred_label} {correct}")

    # Ekspordi MusicXML
    export_attention_musicxml(path, result, output_dir)

    all_results[name] = {
        'true': true_label,
        'pred': pred_label,
        'prob': result['probs'][0][result['predicted_class']].item(),
        'correct': pred_label == true_label
    }

# Kokkuvõte
print(f"\n{'='*60}")
print("KOKKUVÕTE")
print('='*60)
correct_count = sum(1 for r in all_results.values() if r['correct'])
print(f"Õigesti: {correct_count}/{len(all_results)}\n")

for name, r in all_results.items():
    mark = "✓" if r['correct'] else "✗"
    print(f"  {mark} {name:30s} tegelik={r['true']:6s} ennustus={r['pred']:6s} ({r['prob']*100:.0f}%)")

print(f"\nRollout failid salvestatud: {output_dir}")

Eesti faile saadaval: 77
Soome faile saadaval: 702

Eesti: 22.xml
  [HOIATUS] Eemaldan vigased kordusmärgid...


100%|██████████| 1/1 [00:00<00:00, 85.94it/s]


Ennustus: Eesti (90.8%)
   Eesti:  90.8% ███████████████████████████
   Soome:   9.2% ██
Tokeneid (mitte-PAD): 174
  Tegelik: Eesti | Ennustus: Eesti ✓

Töötlen: 22.xml
Joondatud: 174/174 tokenit → 174/175 originaalnooti
Harmony sümbolid eemaldatud (REMOVE_HARMONY=True)
Värvitud: 13 värvilist, 161 musta, 1 halli
Velocity vahemik: 30–127
Salvestatud: /content/drive/MyDrive/LÕPUTÖÖ/rollout_tulemused/22_rollout.musicxml

Eesti: 12.xml


100%|██████████| 1/1 [00:00<00:00, 66.07it/s]


Ennustus: Eesti (98.5%)
   Eesti:  98.5% █████████████████████████████
   Soome:   1.5% 
Tokeneid (mitte-PAD): 250
  Tegelik: Eesti | Ennustus: Eesti ✓

Töötlen: 12.xml
Joondatud: 250/250 tokenit → 250/252 originaalnooti
Harmony sümbolid eemaldatud (REMOVE_HARMONY=True)
Värvitud: 25 värvilist, 212 musta, 2 halli
Velocity vahemik: 30–127
Salvestatud: /content/drive/MyDrive/LÕPUTÖÖ/rollout_tulemused/12_rollout.musicxml

Eesti: 41.xml


...

Ennustus: Soome (95.3%)
   Eesti:   4.7% █
   Soome:  95.3% ████████████████████████████
Tokeneid (mitte-PAD): 100
  Tegelik: Soome | Ennustus: Soome ✓

Töötlen: kt1_0243.mid
Joondatud: 100/100 tokenit → 100/100 originaalnooti
Harmony sümbolid eemaldatud (REMOVE_HARMONY=True)
Värvitud: 52 värvilist, 48 musta, 0 halli
Velocity vahemik: 30–127
Salvestatud: /content/drive/MyDrive/LÕPUTÖÖ/rollout_tulemused/kt1_0243_rollout.musicxml

Soome: kt1_0221.mid


100%|██████████| 1/1 [00:00<00:00, 76.46it/s]


Ennustus: Soome (59.4%)
   Eesti:  40.6% ████████████
   Soome:  59.4% █████████████████
Tokeneid (mitte-PAD): 140
  Tegelik: Soome | Ennustus: Soome ✓

Töötlen: kt1_0221.mid
Joondatud: 140/140 tokenit → 140/140 originaalnooti
Harmony sümbolid eemaldatud (REMOVE_HARMONY=True)
Värvitud: 78 värvilist, 62 musta, 0 halli
Velocity vahemik: 30–127
Salvestatud: /content/drive/MyDrive/LÕPUTÖÖ/rollout_tulemused/kt1_0221_rollout.musicxml

Soome: kt1_0135.mid


100%|██████████| 1/1 [00:00<00:00, 138.01it/s]


Ennustus: Soome (99.6%)
   Eesti:   0.4% 
   Soome:  99.6% █████████████████████████████
Tokeneid (mitte-PAD): 96
  Tegelik: Soome | Ennustus: Soome ✓

Töötlen: kt1_0135.mid
Joondatud: 96/96 tokenit → 96/96 originaalnooti
Harmony sümbolid eemaldatud (REMOVE_HARMONY=True)
Värvitud: 96 värvilist, 0 musta, 0 halli
Velocity vahemik: 30–127
Salvestatud: /content/drive/MyDrive/LÕPUTÖÖ/rollout_tulemused/kt1_0135_rollout.musicxml

KOKKUVÕTE
Õigesti: 6/6

  ✓ 22.xml                         tegelik=Eesti  ennustus=Eesti  (91%)
  ✓ 12.xml                         tegelik=Eesti  ennustus=Eesti  (98%)
  ✓ 41.xml                         tegelik=Eesti  ennustus=Eesti  (99%)
  ✓ kt1_0243.mid                   tegelik=Soome  ennustus=Soome  (95%)
  ✓ kt1_0221.mid                   tegelik=Soome  ennustus=Soome  (59%)
  ✓ kt1_0135.mid                   tegelik=Soome  ennustus=Soome  (100%)

Rollout failid salvestatud: /content/drive/MyDrive/LÕPUTÖÖ/rollout_tulemused
